# 3. STAC interoperability and coverage analysis

The backend *is* a STAC catalog, so standard tooling works with or without this
client. This notebook also looks at what the catalog actually covers.

```bash
pip install 'open-sar-triad[notebooks]'
```

In [ ]:
# Point at the hosted API, or a local build for testing.
#   BASE = 'http://localhost:8000/api/v1'   # after: python3 -m http.server 8000
BASE = None

from opensartriad import Catalog

cat = Catalog(BASE) if BASE else Catalog()
cat

## Into pandas

`to_dataframe()` gives the index-level fields, which is enough for most analysis
and does not trigger the lazy per-provider fetch.

In [ ]:
df = cat.all().to_dataframe()
print(df.shape)
df.head()

In [ ]:
df.groupby('provider').size().sort_values(ascending=False)

## Acquisitions over time

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

monthly = df.assign(month=df['date'].str[:7]).groupby(['month', 'provider']).size().unstack(fill_value=0)

ax = monthly.plot(kind='area', figsize=(11, 4), linewidth=0, alpha=.85)
ax.set_title('Open SAR acquisitions per month')
ax.set_xlabel('')
ax.set_ylabel('scenes')
plt.tight_layout()
plt.show()

## Sensor modes by provider

In [ ]:
pivot = df.groupby(['provider', 'mode']).size().unstack(fill_value=0)
pivot

## Where the scenes are

Each scene carries a bounding box, so a scatter of centroids is a quick way to see
global coverage without any geospatial dependencies.

In [ ]:
cx = (df['west'] + df['east']) / 2
cy = (df['south'] + df['north']) / 2

fig, ax = plt.subplots(figsize=(12, 5.5))
for p, colour in [('umbra', '#00C9FF'), ('capella', '#FF6B35'), ('iceye', '#00FF87')]:
    m = df['provider'] == p
    ax.scatter(cx[m], cy[m], s=5, alpha=.45, label=f'{p} ({m.sum()})', c=colour)

ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title('Scene centroids')
ax.grid(alpha=.15)
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

## Which products are available

How many scenes can satisfy each family. This is the honest answer to
'can I get complex data here?'

In [ ]:
from opensartriad import FAMILIES

rows = []
for fam in FAMILIES:
    hits = cat.search(family=fam)
    by_p = {}
    for s in hits:
        by_p[s.provider] = by_p.get(s.provider, 0) + 1
    rows.append({'family': fam, 'total': len(hits), **by_p})

import pandas as pd
pd.DataFrame(rows).set_index('family').fillna(0).astype(int)

## STAC interoperability

Search results convert straight to a STAC ItemCollection.

In [ ]:
sel = cat.search(bbox=(5.9, 47.2, 10.5, 55.1), start='2025-01-01', limit=5)
ic = sel.to_stac()

print(ic['type'], '| stac_version', ic['stac_version'], '|', len(ic['features']), 'items')
item = ic['features'][0]
print('id      :', item['id'])
print('assets  :', list(item['assets'])[:6])
print('bbox    :', item['bbox'])

### Using pystac directly

No client needed. Any STAC tool can read the catalog by URL.

```python
import pystac

root = pystac.Catalog.from_file(
    'https://www.pmuguda.com/open-sar-triad/api/v1/catalog.json'
)
for child in root.get_children():
    print(child.id, '-', child.title)
```

## Licence

Scene metadata is CC-BY 4.0. Credit the originating provider in published work.

In [ ]:
print(cat.license()['attribution'])